In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())


In [ ]:
# Set browser user agent
# headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0'}

# url = "https://www.bls.gov/cew/classifications/areas/qcew-county-msa-csa-crosswalk.xlsx"

request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 2, engine='openpyxl')
df

In [ ]:
# Extract State Abbreviation from MSA label
def extract_state(text):
    return text.split(',')[1].strip()

df2 = df.copy()
df2.loc[:, 'State'] = df2['MSA Title'].apply(extract_state).str[:2]
df2.loc[:, 'County Code'] = df2['County Code'].astype(str).apply(lambda s : s[-3:])
df2.loc[:, 'MSA Code'   ] = df2['MSA Code'   ].apply(lambda s : s[1:] + '0')
df2 = df2.drop(['County Title', 'CSA Code', 'CSA Title', 'MSA Title'], axis = 1)
df2 = df2.rename(columns = {'County Code':'County FIPS', 'MSA Code':'MSA_ID'})
df2 = df2.sort_values('County FIPS')
df2

In [ ]:
df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips = df_fips.merge(df2, on = ['State', 'County FIPS'], how = 'left')
df_fips

In [ ]:
df_fips[df_fips.duplicated(['State', 'County FIPS'])]

df_fips[(df_fips['State'] == 'AR') & (df_fips['County FIPS'] == '079')]

In [ ]:
df2[df2['MSA_ID'].isin(['38220', '22900'])]